In [19]:
import ast
from datasets import load_dataset
import pandas as pd

### Math afrimgsm

In [20]:
data_english_math = load_dataset('masakhane/afrimgsm', 'eng') 
data_yor_math = load_dataset('masakhane/afrimgsm', "yor") 

In [21]:
data_english_math, data_yor_math

(DatasetDict({
     train: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 8
     })
     test: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 250
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 8
     })
     test: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 250
     })
 }))

In [22]:
data_yor_math["test"]["equation_solution"]

Column([None, None, None, None, None, ...])

In [23]:
data_yor_math["train"]["answer"][0]

'Ìdáhùn lẹ́sẹsẹ: Roger bẹ̀rẹ̀ pẹ̀lú bọ́ọ́lù 5. Agolo 2 bọ́ọ́lù aláfajọ̀gbá kọ̀ọ̀kan 3 jẹ́ bọ́ọ́lù aláfajọ̀gbá 6. 5 + 6 = 11. Ìdáhùn náà ni 11.'

In [24]:
rag_rows = []

In [25]:


for split in ["train", "test"]:
    number_rows = len(data_english_math[split]["question"])
    
    for index in range(number_rows):
        q_yor = data_yor_math[split]["question"][index]
        q_english = data_english_math[split]["question"][index]
        
        ans_yor = data_yor_math[split]["answer"][index]
        
        ans_num = data_yor_math[split]["answer_number"][index]
        eq_english = data_yor_math[split]["equation_solution"][index]
        
        
        context = f"{q_yor} | {q_english}"
        
        
        if ans_yor is not None and eq_english is not None:
            answer = (
                f"[Verified Translation]: {q_english}\n"
                f"[Yorùbá Step-by-Step]: {ans_yor}\n"
                f"[Math Equation]: {eq_english}\n"
                f"[Exact Final Answer]: {ans_num}"
            )
        
        else:
            answer = (
                f"[Verified Parallel Math Problem (Yorùbá -> English)]:\n"
                f"- Yorùbá Text: {q_yor}\n"
                f"- Exact English Meaning: {q_english}\n"
                f"- Verified Numerical Result: {ans_num}"
            )
        
        rag_rows.append({
            "language": "yor",
            "source_dataset": f"afrimgsm_{split}",
            "search_context": context,
            "pedagogical_answer": answer
        })



### Multiple question for general intelligence (afrimmlu)

In [26]:
data_multiple_questions_eng = load_dataset('masakhane/afrimmlu', 'eng')
data_multiple_questions_yor = load_dataset('masakhane/afrimmlu', 'yor')

In [27]:
data_multiple_questions_eng 

DatasetDict({
    validation: Dataset({
        features: ['subject', 'question', 'choices', 'answer'],
        num_rows: 83
    })
    dev: Dataset({
        features: ['subject', 'question', 'choices', 'answer'],
        num_rows: 25
    })
    test: Dataset({
        features: ['subject', 'question', 'choices', 'answer'],
        num_rows: 500
    })
})

### Looking for Nan's


In [28]:
print(data_multiple_questions_eng["validation"]["subject"].count(None) , data_multiple_questions_eng["dev"]["subject"].count(None) , data_multiple_questions_eng["test"]["subject"].count(None))
print(data_multiple_questions_eng["validation"]["question"].count(None) , data_multiple_questions_eng["dev"]["question"].count(None) , data_multiple_questions_eng["test"]["question"].count(None))
print(data_multiple_questions_eng["validation"]["choices"].count(None) , data_multiple_questions_eng["dev"]["choices"].count(None) , data_multiple_questions_eng["test"]["choices"].count(None))

0 0 0
0 0 0
0 0 0


In [29]:
for split in ["validation", "dev", "test"]:
    number_rows = len(data_multiple_questions_eng[split]["question"])
    for index in range(number_rows):
        q_yor = data_multiple_questions_yor[split]["question"][index]
        q_english = data_multiple_questions_eng[split]["question"][index]
        
        eng_sub = data_multiple_questions_eng[split]["subject"][index]
        
        yor_answer = data_multiple_questions_yor[split]["answer"][index]
        eng_anwser = data_multiple_questions_eng[split]["answer"][index]
        
        assert data_multiple_questions_eng[split]["subject"][index] == data_multiple_questions_yor[split]["subject"][index], "Must be the same"
        assert data_multiple_questions_eng[split]["answer"][index] == data_multiple_questions_yor[split]["answer"][index], "Must be the same"
        
        # choices comes as a string, e.g. "['p = 4', 'p = 8', 'p = 12', 'p = 24']"
        choices_yor = ast.literal_eval(data_multiple_questions_yor[split]["choices"][index])
        choices_eng = ast.literal_eval(data_multiple_questions_eng[split]["choices"][index])
        answer_index = "ABCD".index(yor_answer)
        
        options_yor = " | ".join(f"{letter}) {choice}" for letter, choice in zip("ABCD", choices_yor))
        options_eng = " | ".join(f"{letter}) {choice}" for letter, choice in zip("ABCD", choices_eng))
                
        context = f"{q_yor} | {q_english} | {eng_sub}"
        
        answer =  (
                f"[Subject: {eng_sub} | Academic Bilingue Reference]:\n"
                f"- Yorùbá Question: {q_yor}\n"
                f"- English Question: {q_english}\n"
                f"- Options (Yorùbá): {options_yor}\n"
                f"- Options (English): {options_eng}\n"
                f"- Verified Answer (Yorùbá): {yor_answer}) {choices_yor[answer_index]}\n"
                f"- Verified Answer (English): {eng_anwser}) {choices_eng[answer_index]}"
            )
        
        rag_rows.append({
            "language": "yor",
                "source_dataset": f"afrimmlu_{split}",
                "search_context": context,
                "pedagogical_answer": answer
            })

### Lecture comprehesion

In [30]:
data_reading_comphresion_yor = load_dataset('facebook/belebele', "yor_Latn")
data_reading_comphresion_english = load_dataset('facebook/belebele', "eng_Latn")

In [31]:
data_reading_comphresion_yor["test"]["correct_answer_num"]

Column(['1', '1', '2', '2', '3', ...])

In [32]:
data_reading_comphresion_english

DatasetDict({
    test: Dataset({
        features: ['link', 'question_number', 'flores_passage', 'question', 'mc_answer1', 'mc_answer2', 'mc_answer3', 'mc_answer4', 'correct_answer_num', 'dialect', 'ds'],
        num_rows: 900
    })
})

In [33]:
for split in ["test"]:
    # 1. Indexamos el split de inglés por su clave única: (link, question_number)
    eng_lookup = {}
    for i in range(len(data_reading_comphresion_english[split]["question"])):
        key = (
            data_reading_comphresion_english[split]["link"][i],
            data_reading_comphresion_english[split]["question_number"][i]
        )
        eng_lookup[key] = i

    
    number_rows = len(data_reading_comphresion_yor[split]["question"])
    for index in range(number_rows):
        key = (
            data_reading_comphresion_yor[split]["link"][index],
            data_reading_comphresion_yor[split]["question_number"][index]
        )
        
        if key not in eng_lookup:
            continue
            
        eng_idx = eng_lookup[key]
        
        
        yor_q = data_reading_comphresion_yor[split]["question"][index]
        eng_q = data_reading_comphresion_english[split]["question"][eng_idx]
        
        yor_passage = data_reading_comphresion_yor[split]["flores_passage"][index]
        eng_passage = data_reading_comphresion_english[split]["flores_passage"][eng_idx]
        
        
        correct_answer_yor = str(data_reading_comphresion_yor[split]["correct_answer_num"][index]).strip()
        correct_answer_eng = str(data_reading_comphresion_english[split]["correct_answer_num"][eng_idx]).strip()
        
        mac_selection_yor = data_reading_comphresion_yor[split][f"mc_answer{correct_answer_yor}"][index]
        mac_selection_eng = data_reading_comphresion_english[split][f"mc_answer{correct_answer_eng}"][eng_idx]
        
        context = f"{yor_q} | {eng_q}"
        
        answer = (
            f"[Reading Passage (Yorùbá)]: {yor_passage}\n"
            f"[Reading Passage (English)]: {eng_passage}\n"
            f"[Verified Answer (Yorùbá)]: {mac_selection_yor}\n"
            f"[Verified Answer (English)]: {mac_selection_eng}"
        )
    
        rag_rows.append({
            "language": "yor",
            "source_dataset": f"belebele_{split}",
            "search_context": context,
            "pedagogical_answer": answer
        })

In [34]:
df_african = pd.DataFrame(rag_rows)

In [35]:
print(df_african["source_dataset"].value_counts())
df_african.to_csv("dataset_rag.csv", index=False)

source_dataset
belebele_test          900
afrimmlu_test          500
afrimgsm_test          250
afrimmlu_validation     83
afrimmlu_dev            25
afrimgsm_train           8
Name: count, dtype: int64
